In [1]:
#!pip install emoji

In [3]:
import pandas as pd
import re
import emoji as emoji_lib
from collections import Counter

In [ ]:
# ================= CONFIG =================
INPUT_FILE = "input.csv"
OUTPUT_FILE = "cleaned_output.csv"

TEXT_COL = "text"
LABEL_COL = "label"

In [ ]:
# =========================================
# LANGUAGE DETECTION 🔥
# =========================================
def detect_type(text):
    if not isinstance(text, str):
        return "Unknown"

    has_hindi = bool(re.search(r'[\u0900-\u097F]', text))
    has_english = bool(re.search(r'[a-zA-Z]', text))

    if has_hindi and has_english:
        return "Hinglish"
    elif has_hindi:
        return "Hindi"
    elif has_english:
        return "English"
    return "Other"

In [ ]:
import re
import emoji

def remove_emojis(text):
    text = str(text)

    # remove all emojis
    text = emoji.replace_emoji(text, replace='')

    # fix spacing
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
def is_only_emojis(text):
    return remove_emojis(text).strip() == ""

def has_enough_text(text, min_length):
    text = remove_emojis(text)
    ascii_only = re.sub(r'[^a-zA-Z\u0900-\u097F]', '', text)
    return len(ascii_only) >= min_length and len(text.split()) >= 3

def is_valid(text):
    return isinstance(text, str) and len(text.strip().split()) >= 3

In [ ]:
# =========================================
# COMMON CLEANING
# =========================================
def basic_clean(text):
    text = text.lower()

    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'#\S+', '', text)

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# =========================================
# LANGUAGE-SPECIFIC CLEANING 🔥
# =========================================

# 1. ENGLISH
# English

def process_english(df):
    df = df[~df[TEXT_COL].apply(is_only_emojis)]
    df = df[df[TEXT_COL].apply(lambda x: has_enough_text(x, 5))]

    df[TEXT_COL] = df[TEXT_COL].apply(clean_comment)
    df[TEXT_COL] = df[TEXT_COL].apply(remove_emojis)

    df = df.drop_duplicates(subset=TEXT_COL).reset_index(drop=True)

    df[TEXT_COL] = df[TEXT_COL].str.replace(r'[^a-zA-Z\s]', '', regex=True)
    df[TEXT_COL] = df[TEXT_COL].str.replace(r'\s+', ' ', regex=True).str.strip()

    df = df[df[TEXT_COL].apply(is_valid)]

    return df


In [ ]:
# 2. HINDI (Devanagari)

def process_hindi(df):
    df = df[~df[TEXT_COL].apply(is_only_emojis)]
    df = df[df[TEXT_COL].apply(lambda x: has_enough_text(x, 5))]

    df[TEXT_COL] = df[TEXT_COL].apply(clean_comment)
    df[TEXT_COL] = df[TEXT_COL].apply(remove_emojis)

    df[TEXT_COL] = df[TEXT_COL].str.replace(r'[^\u0900-\u097F\s]', '', regex=True)

    df = df.drop_duplicates(subset=TEXT_COL).reset_index(drop=True)

    df[TEXT_COL] = df[TEXT_COL].str.replace(r'\s+', ' ', regex=True).str.strip()

    df = df[df[TEXT_COL].apply(is_valid)]

    return df


In [ ]:
# 3. HINGLISH
def process_hinglish(df):
    df = df[~df[TEXT_COL].apply(is_only_emojis)]
    df = df[df[TEXT_COL].apply(lambda x: has_enough_text(x, 7))]
    df = df[df[TEXT_COL].apply(lambda x: has_enough_text(x, 14))]

    df[TEXT_COL] = df[TEXT_COL].apply(clean_comment)
    df[TEXT_COL] = df[TEXT_COL].apply(remove_emojis)

    df = df[df[TEXT_COL].apply(is_valid)]
    df = df.drop_duplicates(subset=TEXT_COL).reset_index(drop=True)

    df[TEXT_COL] = df[TEXT_COL].str.replace(r'[£€@#*!]+', '', regex=True)
    df[TEXT_COL] = df[TEXT_COL].str.replace(r'(?<!\w)\b\w\b(?!\w)', '', regex=True)
    df[TEXT_COL] = df[TEXT_COL].str.replace(r'\s+', ' ', regex=True).str.strip()

    return df

In [ ]:
# =========================================
# LOAD
# =========================================
df = pd.read_csv(INPUT_FILE)
df = df[[TEXT_COL, LABEL_COL]]

print("Original:", df.shape)

In [ ]:
# =========================================
# DETECT TYPES
# =========================================
df["Type"] = df["Comment"].apply(detect_type)


In [ ]:
# =========================================
# SPLIT + PROCESS
# =========================================
hinglish_df = process_hinglish(df[df["Type"] == "Hinglish"].copy())
english_df  = process_english(df[df["Type"] == "English"].copy())
hindi_df    = process_hindi(df[df["Type"] == "Hindi"].copy())

In [ ]:
# =========================================
# MERGE
# =========================================
final_df = pd.concat([hinglish_df, english_df, hindi_df]).reset_index(drop=True)


In [ ]:
# =========================================
# LABEL CLEAN
# =========================================
final_df[LABEL_COL] = final_df[LABEL_COL].astype(str)
final_df[LABEL_COL] = final_df[LABEL_COL].str.replace("'", "")

In [ ]:
df["Label"] = df["Label"].str.replace("'", "", regex=False)
df["Label"] = df["Label"].map({'HS0': 0, 'HS1': 1, 'HSN': 1})